# W03 - Data Quality and Issues

## Real-World Example: Customer Data Quality Analysis

In this notebook, we will create a sample dataset with deliberate data quality issues and walk through the detection and cleaning process step by step.

### The Scenario

Imagine you are a data scientist at an e-commerce company. You receive customer data from:
- Online registration forms
- Customer support entries
- Legacy database imports
- Third-party data enrichment

The combined data has multiple quality issues that need to be addressed before any analysis or machine learning.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")

## Step 1: Creating a Sample Dataset with Artificial Data Quality Issues

We will deliberately create a dataset with multiple quality problems:
1. **Missing Values (Incomplete Data)**
2. **Noisy Data** (incorrect values due to errors)
3. **Inconsistent Formats** (dates, categories)
4. **Outliers** (extreme but possibly valid values)
5. **Duplicate Records**

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)
n_customers = 200

# Create base clean data
customer_ids = range(1001, 1001 + n_customers)
ages = np.random.normal(35, 12, n_customers).astype(int)
ages = np.clip(ages, 18, 80)  # Keep ages realistic

incomes = np.random.normal(50000, 20000, n_customers).astype(int)
incomes = np.clip(incomes, 15000, 200000)

cities = np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Kolkata'], n_customers)

# Create dates with consistent format
from datetime import datetime, timedelta
base_date = datetime(2023, 1, 1)
registration_dates = [base_date + timedelta(days=np.random.randint(0, 365)) for _ in range(n_customers)]

# Create base DataFrame
df_clean = pd.DataFrame({
    'customer_id': customer_ids,
    'age': ages,
    'income': incomes,
    'city': cities,
    'registration_date': registration_dates,
    'purchase_amount': np.random.gamma(2, 500, n_customers).astype(int)
})

print("Clean dataset created successfully!")
print(f"Shape: {df_clean.shape}")
print("\nFirst 5 rows of clean data:")
df_clean.head()

In [ ]:
# NOW INTRODUCE DATA QUALITY ISSUES
df_dirty = df_clean.copy()

print("=" * 60)
print("INTRODUCING DATA QUALITY ISSUES")
print("=" * 60)

# Issue 1: Missing Values (Incomplete Data)
print("\n1. Adding missing values...")
missing_age_idx = np.random.choice(df_dirty.index, 15, replace=False)
df_dirty.loc[missing_age_idx, 'age'] = np.nan

missing_income_idx = np.random.choice(df_dirty.index, 20, replace=False)
df_dirty.loc[missing_income_idx, 'income'] = np.nan

missing_city_idx = np.random.choice(df_dirty.index, 8, replace=False)
df_dirty.loc[missing_city_idx, 'city'] = np.nan

print(f"  - Added {len(missing_age_idx)} missing ages")
print(f"  - Added {len(missing_income_idx)} missing incomes")
print(f"  - Added {len(missing_city_idx)} missing cities")

# Issue 2: Noisy/Incorrect Data
print("\n2. Adding noisy/incorrect values...")
noisy_age_idx = np.random.choice(df_dirty.index, 5, replace=False)
df_dirty.loc[noisy_age_idx, 'age'] = [150, -5, 999, 0, 200]  # Impossible ages
print(f"  - Added {len(noisy_age_idx)} impossible ages (e.g., 150, -5, 999)")

noisy_income_idx = np.random.choice(df_dirty.index, 8, replace=False)
df_dirty.loc[noisy_income_idx, 'income'] = [-10000, 9999999, -5000, 10000000, -1, 88888888, 0, -999]
print(f"  - Added {len(noisy_income_idx)} invalid incomes (negatives, extreme values)")

# Issue 3: Inconsistent Formats
print("\n3. Adding inconsistent formats...")
inconsistent_dates_idx = np.random.choice(df_dirty.index, 10, replace=False)
for idx in inconsistent_dates_idx[:5]:
    df_dirty.loc[idx, 'registration_date'] = '31/12/2023'  # DD/MM/YYYY format
for idx in inconsistent_dates_idx[5:]:
    df_dirty.loc[idx, 'registration_date'] = '2023-15-01'  # Invalid date
print(f"  - Added {len(inconsistent_dates_idx)} inconsistent date formats")

inconsistent_cities_idx = np.random.choice(df_dirty.index, 6, replace=False)
df_dirty.loc[inconsistent_cities_idx, 'city'] = ['BOM', 'DEL', 'BLR', 'MAA', 'CCU', 'MUMBAI']
print(f"  - Added {len(inconsistent_cities_idx)} inconsistent city name formats")

# Issue 4: Outliers (potentially valid but extreme)
print("\n4. Adding outliers...")
outlier_age_idx = np.random.choice(df_dirty.index, 3, replace=False)
df_dirty.loc[outlier_age_idx, 'age'] = [85, 82, 88]  # Old age but possible
print(f"  - Added {len(outlier_age_idx)} age outliers (85+)")

outlier_income_idx = np.random.choice(df_dirty.index, 3, replace=False)
df_dirty.loc[outlier_income_idx, 'income'] = [250000, 300000, 275000]  # High but possible
print(f"  - Added {len(outlier_income_idx)} income outliers (>250k)")

# Issue 5: Duplicate Records
print("\n5. Adding duplicate records...")
duplicate_rows = df_dirty.iloc[:5].copy()
duplicate_rows['customer_id'] = range(2000, 2005)
df_dirty = pd.concat([df_dirty, duplicate_rows], ignore_index=True)
print(f"  - Added 5 duplicate records (exact copies of first 5 rows)")

print("\n" + "=" * 60)
print(f"Final dirty dataset shape: {df_dirty.shape}")
print("=" * 60)

In [ ]:
# Display the dirty dataset
print("DIRTY DATASET PREVIEW:")
print("-" * 80)
df_dirty.head(20)

In [ ]:
# Check data types and basic info
print("DATASET INFORMATION:")
print("-" * 50)
df_dirty.info()

print("\n\nBASIC STATISTICS (shows issues):")
print("-" * 50)
df_dirty.describe()

## Step 2: Detecting Data Quality Issues

### 2.1 Detecting Missing Values

In [ ]:
# Detect missing values
print("MISSING VALUES ANALYSIS")
print("=" * 50)

missing_counts = df_dirty.isnull().sum()
missing_percentages = (df_dirty.isnull().sum() / len(df_dirty)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percentage': missing_percentages
})

print(missing_df[missing_df['Missing Count'] > 0])

# Visualize missing values
plt.figure(figsize=(10, 6))
sns.heatmap(df_dirty.isnull(), yticklabels=False, cbar=True, cmap='viridis')
plt.title('Missing Values Heatmap', fontsize=14)
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.show()

### 2.2 Detecting Noisy/Invalid Values

In [ ]:
# Define validation rules
print("INVALID VALUES DETECTION")
print("=" * 50)

# Age validation: should be between 0 and 120
invalid_age_mask = (df_dirty['age'] < 0) | (df_dirty['age'] > 120)
invalid_ages = df_dirty[invalid_age_mask & df_dirty['age'].notna()]
print(f"\nInvalid ages (outside 0-120): {len(invalid_ages)}")
if len(invalid_ages) > 0:
    print(invalid_ages[['customer_id', 'age']])

# Income validation: should be between 0 and 1,000,000
invalid_income_mask = (df_dirty['income'] < 0) | (df_dirty['income'] > 1000000)
invalid_incomes = df_dirty[invalid_income_mask & df_dirty['income'].notna()]
print(f"\nInvalid incomes (outside 0-1,000,000): {len(invalid_incomes)}")
if len(invalid_incomes) > 0:
    print(invalid_incomes[['customer_id', 'income']])

# City validation: should be in standard list
valid_cities = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Kolkata']
invalid_city_mask = ~df_dirty['city'].isin(valid_cities) & df_dirty['city'].notna()
invalid_cities = df_dirty[invalid_city_mask]
print(f"\nInvalid city names: {len(invalid_cities)}")
if len(invalid_cities) > 0:
    print(invalid_cities[['customer_id', 'city']].unique())

### 2.3 Detecting Inconsistent Date Formats

In [ ]:
# Check date format consistency
print("DATE FORMAT ANALYSIS")
print("=" * 50)

def is_valid_date(date_val):
    """Check if date is in proper datetime format"""
    if pd.isna(date_val):
        return False
    return isinstance(date_val, (pd.Timestamp, datetime))

date_type_check = df_dirty['registration_date'].apply(lambda x: type(x).__name__)
print("\nDate column data types:")
print(date_type_check.value_counts())

# Find rows with invalid date types
invalid_date_rows = df_dirty[~df_dirty['registration_date'].apply(is_valid_date)]
print(f"\nRows with invalid date format: {len(invalid_date_rows)}")
if len(invalid_date_rows) > 0:
    print(invalid_date_rows[['customer_id', 'registration_date']].head(10))

### 2.4 Detecting Duplicates

In [ ]:
# Detect duplicate records
print("DUPLICATE DETECTION")
print("=" * 50)

duplicate_count = df_dirty.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

if duplicate_count > 0:
    print("\nDuplicate rows:")
    print(df_dirty[df_dirty.duplicated(keep=False)].sort_values('customer_id').head(10))

## Step 3: Data Cleaning and Preprocessing

### 3.1 Handling Missing Values

Several strategies for missing data:
- Remove rows with missing values
- Impute with mean/median/mode
- Forward fill or backward fill
- Use predictive imputation

In [ ]:
# Create a clean copy for processing
df_clean_processed = df_dirty.copy()

print("HANDLING MISSING VALUES")
print("=" * 50)

# Strategy 1: Impute age with median
age_median = df_clean_processed['age'].median()
df_clean_processed['age'] = df_clean_processed['age'].fillna(age_median)
print(f"1. Imputed missing ages with median: {age_median:.0f}")

# Strategy 2: Impute income with mean
income_mean = df_clean_processed['income'].mean()
df_clean_processed['income'] = df_clean_processed['income'].fillna(income_mean)
print(f"2. Imputed missing incomes with mean: {income_mean:.0f}")

# Strategy 3: Impute city with mode
city_mode = df_clean_processed['city'].mode()[0]
df_clean_processed['city'] = df_clean_processed['city'].fillna(city_mode)
print(f"3. Imputed missing cities with mode: {city_mode}")

# Verify no missing values remain
print(f"\nRemaining missing values: {df_clean_processed.isnull().sum().sum()}")

### 3.2 Handling Noisy/Invalid Values

In [ ]:
print("HANDLING INVALID VALUES")
print("=" * 50)

# Fix invalid ages
invalid_age_count_before = ((df_clean_processed['age'] < 0) | (df_clean_processed['age'] > 120)).sum()
print(f"\nInvalid ages before cleaning: {invalid_age_count_before}")

# Replace invalid ages with median
age_median = df_clean_processed['age'].median()
df_clean_processed.loc[(df_clean_processed['age'] < 0) | (df_clean_processed['age'] > 120), 'age'] = age_median
print(f"  Replaced invalid ages with median: {age_median:.0f}")

# Fix invalid incomes
invalid_income_before = ((df_clean_processed['income'] < 0) | (df_clean_processed['income'] > 1000000)).sum()
print(f"\nInvalid incomes before cleaning: {invalid_income_before}")

# Replace invalid incomes with mean
income_mean = df_clean_processed['income'].mean()
df_clean_processed.loc[(df_clean_processed['income'] < 0) | (df_clean_processed['income'] > 1000000), 'income'] = income_mean
print(f"  Replaced invalid incomes with mean: {income_mean:.0f}")

# Fix inconsistent city names
print("\nCity name standardization:")
city_mapping = {
    'BOM': 'Mumbai', 'DEL': 'Delhi', 'BLR': 'Bangalore',
    'MAA': 'Chennai', 'CCU': 'Kolkata', 'MUMBAI': 'Mumbai'
}
df_clean_processed['city'] = df_clean_processed['city'].replace(city_mapping)
print(f"  Standardized {len(city_mapping)} city name variations")

invalid_cities_after = (~df_clean_processed['city'].isin(valid_cities)).sum()
print(f"  Invalid cities after cleaning: {invalid_cities_after}")

### 3.3 Handling Inconsistent Dates

In [ ]:
print("HANDLING INCONSISTENT DATES")
print("=" * 50)

def parse_date_robust(date_val):
    """Try multiple date formats"""
    if pd.isna(date_val):
        return np.nan
    if isinstance(date_val, (pd.Timestamp, datetime)):
        return date_val
    
    date_formats = ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d']
    for fmt in date_formats:
        try:
            return datetime.strptime(str(date_val), fmt)
        except:
            continue
    return np.nan

# Parse dates
invalid_dates_before = df_clean_processed['registration_date'].apply(is_valid_date).sum()
print(f"Valid dates before parsing: {invalid_dates_before}")

df_clean_processed['registration_date'] = df_clean_processed['registration_date'].apply(parse_date_robust)

# Fill any remaining invalid dates with a default
default_date = datetime(2023, 1, 1)
df_clean_processed['registration_date'] = df_clean_processed['registration_date'].fillna(default_date)

valid_dates_after = df_clean_processed['registration_date'].notna().sum()
print(f"Valid dates after parsing: {valid_dates_after}")
print(f"Date column type: {type(df_clean_processed['registration_date'].iloc[0])}")

### 3.4 Removing Duplicates

In [ ]:
print("REMOVING DUPLICATES")
print("=" * 50)

duplicates_before = df_clean_processed.duplicated().sum()
print(f"Duplicates before removal: {duplicates_before}")

df_clean_processed = df_clean_processed.drop_duplicates()

duplicates_after = df_clean_processed.duplicated().sum()
print(f"Duplicates after removal: {duplicates_after}")
print(f"Final dataset shape: {df_clean_processed.shape}")

### 3.5 Outlier Analysis

In [ ]:
print("OUTLIER ANALYSIS")
print("=" * 50)

# Use IQR method to detect outliers
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Age outliers
age_outliers, age_lower, age_upper = detect_outliers_iqr(df_clean_processed, 'age')
print(f"\nAge outliers detected: {len(age_outliers)}")
print(f"  Normal range: [{age_lower:.0f}, {age_upper:.0f}]")
if len(age_outliers) > 0:
    print(f"  Outlier values: {age_outliers['age'].tolist()}")

# Income outliers
income_outliers, inc_lower, inc_upper = detect_outliers_iqr(df_clean_processed, 'income')
print(f"\nIncome outliers detected: {len(income_outliers)}")
print(f"  Normal range: [{inc_lower:.0f}, {inc_upper:.0f}]")
if len(income_outliers) > 0:
    print(f"  Outlier values: {income_outliers['income'].tolist()[:10]}")

## Step 4: Visualizing Before and After Cleaning

In [ ]:
# Compare distributions before and after cleaning
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Age distribution
axes[0, 0].hist(df_dirty['age'].dropna(), bins=30, alpha=0.5, label='Before', color='red')
axes[0, 0].hist(df_clean_processed['age'], bins=30, alpha=0.5, label='After', color='green')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Age Distribution')
axes[0, 0].legend()

# Income distribution
axes[0, 1].hist(df_dirty['income'].dropna(), bins=30, alpha=0.5, label='Before', color='red')
axes[0, 1].hist(df_clean_processed['income'], bins=30, alpha=0.5, label='After', color='green')
axes[0, 1].set_xlabel('Income')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Income Distribution')
axes[0, 1].legend()

# Box plot comparison
before_data = df_dirty['purchase_amount'].dropna()
after_data = df_clean_processed['purchase_amount']
axes[0, 2].boxplot([before_data, after_data], labels=['Before', 'After'])
axes[0, 2].set_ylabel('Purchase Amount')
axes[0, 2].set_title('Purchase Amount Boxplot')

# Missing values comparison
missing_before = df_dirty.isnull().sum().sum()
missing_after = df_clean_processed.isnull().sum().sum()
axes[1, 0].bar(['Before Cleaning', 'After Cleaning'], [missing_before, missing_after], color=['red', 'green'])
axes[1, 0].set_ylabel('Number of Missing Values')
axes[1, 0].set_title('Missing Values Comparison')

# Dataset size comparison
size_before = len(df_dirty)
size_after = len(df_clean_processed)
axes[1, 1].bar(['Before Cleaning', 'After Cleaning'], [size_before, size_after], color=['red', 'green'])
axes[1, 1].set_ylabel('Number of Rows')
axes[1, 1].set_title('Dataset Size After Duplicate Removal')

# City distribution after cleaning
city_counts = df_clean_processed['city'].value_counts()
axes[1, 2].pie(city_counts.values, labels=city_counts.index, autopct='%1.1f%%')
axes[1, 2].set_title('City Distribution After Cleaning')

plt.tight_layout()
plt.show()

## Step 5: Final Clean Dataset Summary

In [ ]:
print("FINAL CLEAN DATASET")
print("=" * 60)
print(f"Shape: {df_clean_processed.shape}")
print(f"Columns: {list(df_clean_processed.columns)}")
print(f"\nMissing values: {df_clean_processed.isnull().sum().sum()}")
print(f"Duplicates: {df_clean_processed.duplicated().sum()}")

print("\n" + "=" * 60)
print("CLEAN DATASET PREVIEW:")
print("-" * 60)
df_clean_processed.head(10)

In [ ]:
print("CLEAN DATASET STATISTICS:")
print("-" * 40)
df_clean_processed.describe()

## Step 6: Garbage In → Garbage Out Demonstration

Let's demonstrate how data quality affects a simple machine learning model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

print("GIGO DEMONSTRATION: Impact of Data Quality on Model Performance")
print("=" * 70)

# Prepare features for modeling
features = ['age', 'income']
target = 'purchase_amount'

# Remove rows with missing target in dirty data
df_dirty_model = df_dirty.dropna(subset=[target])

# For dirty data, we need to handle missing values roughly
df_dirty_model_clean = df_dirty_model.copy()
df_dirty_model_clean['age'] = df_dirty_model_clean['age'].fillna(df_dirty_model_clean['age'].median())
df_dirty_model_clean['income'] = df_dirty_model_clean['income'].fillna(df_dirty_model_clean['income'].median())

# Remove extreme outliers for dirty data
df_dirty_model_clean = df_dirty_model_clean[
    (df_dirty_model_clean['age'].between(0, 120)) & 
    (df_dirty_model_clean['income'].between(0, 1000000))
]

# Train models
results = {}

for name, data in [('Dirty Data (with issues)', df_dirty_model_clean), 
                   ('Clean Data', df_clean_processed)]:
    
    X = data[features]
    y = data[target]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'RMSE': rmse, 'R2': r2, 'n_samples': len(data)}
    
    print(f"\n{name}:")
    print(f"  Samples: {len(data)}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R-squared: {r2:.4f}")

print("\n" + "=" * 70)
print("CONCLUSION: Poor data quality leads to worse model performance!")
print("The clean data produced better predictions (higher R2, lower RMSE).")
print("=" * 70)

## Summary: Key Takeaways

1. **Missing Values** - Can be imputed or removed; always detect first
2. **Noisy Data** - Invalid values need domain-specific validation rules
3. **Inconsistent Formats** - Standardize everything to a common format
4. **Outliers** - May be valid or invalid; investigate before removing
5. **Duplicates** - Remove to avoid bias in analysis
6. **GIGO Principle** - Model quality depends directly on data quality

**The Data Quality Pipeline:**
```
Raw Data → Detect Issues → Clean → Validate → Model Ready
```